# Numerické metody — cvičební notebook

Každý úkol říká: **jaká metoda**, **jaká data**, **co chceš získat**.  
Doplň kód do prázdné buňky, spusť, zkontroluj výsledek.

---

In [ ]:
import sys, os, math

BASE = os.path.abspath('.')

sys.path.insert(0, BASE)
sys.path.insert(0, os.path.join(BASE, '01_nelinearni_rovnice'))
sys.path.insert(0, os.path.join(BASE, '02_polynomy_a_interpolace'))
sys.path.insert(0, os.path.join(BASE, '03_soustavy_linearnich_rovnic'))
sys.path.insert(0, os.path.join(BASE, '04_aproximace'))
sys.path.insert(0, os.path.join(BASE, '06_numericka_integrace'))
sys.path.insert(0, os.path.join(BASE, '07_diferencialni_rovnice'))

from bisekce          import bisection
from regula_falsi_A   import regula_falsi
from newtonroot       import newton
from steffansen_A     import steffensen
from fixed_point_A    import fixed_point_iteration
from halley_A         import halley
from newton_horner_A  import newton_horner   # koefs od nejnižší mocniny
from lin_interpolace_po_castech_A import linear_interpolation
from gauss            import gauss
from gaussseidel      import gauss_seidel
from jacobi           import jacobi
from mnc              import lsa
from simpson          import simpson_rule
from romberg_A        import romberg_quadrature
from euler            import euler_step
from runge_kutta_A    import runge_kutta_4

print('Importy OK')

---
## 1. Nelineární rovnice
### 1.1 Bisekce

**Kdy:** máš interval [a,b] kde f(a)·f(b) < 0, chceš spolehlivou (ale pomalou) metodu.  
**Konvergence:** lineární — každá iterace zmenší interval na polovinu.

**Úkol:**  
Najdi kořen funkce `f(x) = x³ - x - 2` na intervalu `[1, 2]`.  
Použij toleranci `1e-8`, max iterací `100`.  
Ověř: `f(koren) ≈ 0`.

In [ ]:
# --- Bisekce ---


import math

def bisection(f, a, b, tol, max_iter):
    """
    Numerické hledání kořene funkce pomocí metody půlení intervalu.
    
    :param f: Funkce, jejíž kořen hledáme
    :param a: Levá mez intervalu
    :param b: Pravá mez intervalu
    :param tol: Tolerance (přesnost)
    :param max_iter: Maximální počet iterací
    :return: Kořen funkce nebo None, pokud metoda selže
    """
    fa = f(a)
    fb = f(b)

    # Kontrola, zda v intervalu vůbec může být kořen (změna znaménka)
    if fa * fb > 0:
        print("Error: No sign change, can't guarantee a root.")
        return None

    c = a
    for i in range(max_iter):
        c = (a + b) / 2
        fc = f(c)

        # Kontrola, zda jsme dosáhli dostatečné přesnosti
        if abs(fc) < tol or (b - a) / 2 < tol:
            return c

        # Rozhodnutí, kterou polovinu intervalu si ponecháme
        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc

    print("Error: Did not converge.")
    return None

# Příklad použití:
# root = bisection(lambda x: x**2 - 4, 0, 5, 1e-6, 100)
# print(f"Kořen je: {root}")


function_for_bis = lambda x: x**3 - x - 2

# TODO: zavolej bisection(f, a, b, tol, max_iter)

koren = bisection(function_for_bis,1,2,1e-6,100)


if koren != None:
    print(f"Funkce má kořen(polovinu) v bodě {koren:4f}")
else:
    print(f"Funkce nemá kořen, f(a)*f(b) nikdy není menší jak nula a nedochází ke změně znaménka")
















Funkce má kořen(polovinu) v bodě 1.521380


In [ ]:
# ŘEŠENÍ
f = lambda x: x**3 - x - 2
koren = bisection(f, 1, 2, 1e-8, 100)
print(f'koren = {koren:.10f}')         # ≈ 1.5213797068
print(f'f(koren) = {f(koren):.2e}')

### 1.2 Regula Falsi

**Kdy:** stejné podmínky jako bisekce, ale nový bod je průsečík sečny → rychlejší u nelineárních f.  
**Nevýhoda:** jeden konec intervalu může zůstat zafixovaný.

**Úkol:**  
Najdi kořen `f(x) = e^x - 3x` na intervalu `[1, 2]`.  
Tolerance `1e-9`, max `200` iterací.

In [18]:



import math

def regula_falsi(f, a, b, tol, max_iter):
    """
    Hledání kořene rovnice f(x)=0 metodou regula falsi (sečnová metoda s intervalem).
    Rozdíl od bisekce: nový bod c je průsečík sečny, ne střed intervalu.

    :param f: Funkce, jejíž kořen hledáme
    :param a: Levá mez intervalu
    :param b: Pravá mez intervalu
    :param tol: Tolerance (přesnost)
    :param max_iter: Maximální počet iterací
    :return: Odhad kořene nebo None při selhání metody
    """
    fa = f(a)
    fb = f(b)

    if fa * fb > 0:
        print("Error: No sign change, can't guarantee a root.")
        return None

    c = a
    for i in range(max_iter):
        if abs(fb - fa) < 1e-12:
            print("Error: Zero denominator (f(a) == f(b)).")
            return None

        c = a - fa * (b - a) / (fb - fa)
        fc = f(c)

        if math.isnan(fc) or math.isinf(fc):
            print("Error: Did not converge (NaN/Inf).")
            return None

        if abs(fc) < tol:
            return c

        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc

    print("Error: Did not converge within max iterations.")
    return None




# --- Regula Falsi ---
fun = lambda x: math.exp(x) - 3*x

vysledek = regula_falsi(fun,1,2, 1e-9, 100)

print(vysledek)








1.5121345512034226


In [ ]:
# ŘEŠENÍ
f = lambda x: math.exp(x) - 3*x
koren = regula_falsi(f, 1, 2, 1e-9, 200)
print(f'koren = {koren:.10f}')    # ≈ 1.5121345516
print(f'f(koren) = {f(koren):.2e}')

### 1.3 Newton-Raphson

**Kdy:** znáš derivaci, máš dobrý počáteční odhad. Kvadratická konvergence.  
**Riziko:** diverguje pokud f'(x) ≈ 0 nebo špatný x0.

**Úkol:**  
Najdi kořen `f(x) = cos(x) - x` (slavná rovnice, kořen se nazývá Dottie number).  
Derivace: `f'(x) = -sin(x) - 1`.  
Počáteční odhad `x0 = 0.5`, tolerance `1e-12`, max `50` iterací.

In [33]:
# --- Newton-Raphson ---
# Definuj f(x) = cos(x) - x a její derivaci fd(x)
# Zavolej newton(f, fd, x0, tol, max_iter) s x0=0.5, tol=1e-12, max_iter=50


import math


def newton(f, fd, x0, tol, max_iter, verbose=False):
    """
    Hledání kořene funkce f(x)=0 pomocí Newtonovy (Newton-Raphsonovy) metody.

    :param f: Funkce, jejíž kořen hledáme
    :param fd: Derivace funkce f
    :param x0: Počáteční odhad kořene
    :param tol: Tolerance (přesnost)
    :param max_iter: Maximální počet iterací
    :param verbose: Vypisovat průběh iterací (default False)
    :return: Odhad kořene nebo None při selhání metody
    """
    x = x0
    for i in range(max_iter):
        fx  = f(x)
        fdx = fd(x)

        if abs(fdx) < 1e-12:
            print("Error: Zero derivative, metoda selhala.")
            return None

        x_new = x - fx / fdx

        if verbose:
            print(f"Iterace {i+1:3d}: x = {x_new:.10f}, f(x) = {f(x_new):.4e}, krok = {abs(x_new-x):.4e}")

        if abs(x_new - x) < tol:
            if verbose:
                print(f"Konvergoval v iteraci {i+1}.")
            return x_new

        x = x_new

    return x



ff = lambda x: math.cos(x) - x           #Funkce
fdd = lambda x: -math.sin(x) - 1 
x0= 0.5
tol = 1e-12
iter = 50


vysledek = newton(ff,fdd,x0,tol,iter, verbose=True)

print(vysledek)

Iterace   1: x = 0.7552224171, f(x) = -2.7103e-02, krok = 2.5522e-01
Iterace   2: x = 0.7391416661, f(x) = -9.4615e-05, krok = 1.6081e-02
Iterace   3: x = 0.7390851339, f(x) = -1.1810e-09, krok = 5.6532e-05
Iterace   4: x = 0.7390851332, f(x) = 0.0000e+00, krok = 7.0565e-10
Iterace   5: x = 0.7390851332, f(x) = 0.0000e+00, krok = 0.0000e+00
Konvergoval v iteraci 5.
0.7390851332151607


In [ ]:
# ŘEŠENÍ
f  = lambda x: math.cos(x) - x
fd = lambda x: -math.sin(x) - 1
koren = newton(f, fd, 0.5, 1e-12, 50)
print(f'koren = {koren:.12f}')   # ≈ 0.739085133215
print(f'f(koren) = {f(koren):.2e}')

### 1.4 Steffensenova metoda

**Kdy:** chceš rychlost Newtona ale **bez derivace** — odhaduje ji z hodnot funkce.  
**Konvergence:** kvadratická (jako Newton).

**Úkol:**  
Najdi kořen `f(x) = x² - 5` (tj. √5).  
Počáteční odhad `x0 = 2.0`, tolerance `1e-10`, max `100` iterací.  
Ověř: `koren² ≈ 5`.

In [39]:
# --- Steffensen ---
import math




def steffensen(f, x0, tol, max_iter, verbose=False):
    """
    Hledání kořene funkce f(x)=0 pomocí Steffensenovy metody.

    :param f: Funkce, jejíž kořen hledáme (není třeba derivace)
    :param x0: Počáteční odhad kořene
    :param tol: Tolerance (přesnost)
    :param max_iter: Maximální počet iterací
    :param verbose: Vypisovat průběh iterací (default False)
    :return: Odhad kořene nebo None při selhání metody
    """
    if f is None or x0 is None or tol is None or max_iter is None:
        print("Error: Nil values are not supported.")
        return None
    x = x0
    for i in range(max_iter):
        f_x = f(x)
        f_x1 = f(x + f_x)
        denominator = f_x1 - f_x
        if abs(denominator) < 1e-12:
            print("Error: Zero denominator in Steffensen method.")
            return None
        x_new = x - (f_x * f_x) / denominator
        if math.isnan(x_new) or math.isinf(x_new):
            print("Error: Did not converge (diverged to NaN/Inf).")
            return None

        if verbose:
            print(f"Iterace {i+1:3d}: x = {x_new:.10f}, f(x) = {f(x_new):.4e}, krok = {abs(x_new-x):.4e}")

        if abs(x_new - x) < tol or abs(f(x_new)) < tol:
            if verbose:
                print(f"Konvergoval v iteraci {i+1}.")
            return x_new
        x = x_new
    print("Error: Did not converge within max iterations.")
    return None





f= lambda x: x**2 - 5 
x0 = 2.0
tol = 1e-10
iter= 100

vysledekk = steffensen(f,x0,tol,iter,verbose=True)

print(math.sqrt(5))
print(vysledekk)


print("Můžeme vidět že steffensnova metoda je schopna spočítat odmocninu ")




Iterace   1: x = 2.3333333333, f(x) = 4.4444e-01, krok = 3.3333e-01
Iterace   2: x = 2.2463768116, f(x) = 4.6209e-02, krok = 8.6957e-02
Iterace   3: x = 2.2361963396, f(x) = 5.7407e-04, krok = 1.0180e-02
Iterace   4: x = 2.2360679977, f(x) = 9.0149e-08, krok = 1.2834e-04
Iterace   5: x = 2.2360679775, f(x) = 2.6645e-15, krok = 2.0158e-08
Konvergoval v iteraci 5.
2.23606797749979
2.2360679774997902
Můžeme vidět že steffensnova metoda je schopna spočítat odmocninu 


In [ ]:
# ŘEŠENÍ
f = lambda x: x**2 - 5
koren = steffensen(f, 2.0, 1e-10, 100)
print(f'koren = {koren:.12f}')   # ≈ 2.236067977500
print(f'koren² = {koren**2:.10f}')

### 1.5 Metoda pevného bodu

**Kdy:** přepíšeš `f(x)=0` na tvar `x = φ(x)` a φ je kontrakce (|φ'(x)| < 1 v okolí kořene).  
**Konvergence:** lineární.

**Úkol:**  
Rovnice: `x³ - x - 1 = 0` → přepíšeme na `x = (x+1)^(1/3)` → `φ(x) = (x+1)^(1/3)`.  
Počáteční odhad `x0 = 1.5`, tolerance `1e-8`, max `200` iterací.

In [42]:
# --- Fixed Point ---
# Rovnice x³ - x - 1 = 0 → přepiš na tvar x = φ(x)
# Zavolej fixed_point_iteration(phi, x0, tol, max_iter) s x0=1.5, tol=1e-8, max_iter=200


# x**3 -x - 1 = 0     #chceme prepsat na rovnici typu x=

#x**3 = x + 1  #přehodili jsme na stranu x na třetí a ted odmocnime 

#x = math.sqrt(x + 1) # můžeme vyjádřit jako ikdyž je to asi jako 

#x = (x+1)**(1/3)   #jen přepis odmocniny do zlomku 


import math

def fixed_point_iteration(phi, x0, tol, max_iter, verbose=False):
    """
    Hledání pevného bodu funkce phi metodou prosté iterace.

    :param phi: Iterační funkce g(x), jejíž pevný bod hledáme
    :param x0: Počáteční odhad řešení
    :param tol: Tolerance (přesnost)
    :param max_iter: Maximální počet iterací
    :param verbose: Vypisovat průběh iterací (default False)
    :return: Odhad pevného bodu nebo None, pokud iterace selže
    """
    if phi is None or x0 is None or tol is None or max_iter is None:
        print("Error: Nil values are not supported.")
        return None
    x = x0
    for i in range(max_iter):
        x_new = phi(x)
        if math.isnan(x_new) or math.isinf(x_new):
            print("Error: Did not converge (diverged to NaN/Inf).")
            return None

        if verbose:
            print(f"Iterace {i+1:3d}: x = {x_new:.10f}, změna = {abs(x_new-x):.4e}")

        if abs(x_new - x) < tol:
            if verbose:
                print(f"Konvergoval v iteraci {i+1}.")
            return x_new
        x = x_new
    print("Error: Did not converge within max iterations.")
    return None



fí = lambda x: (x+1)**(1/3)
x0 = 1.5 
tol = 1e-8
iter = 200

vysledek = fixed_point_iteration(fí, x0, tol, iter,verbose=True)

print(vysledek)






Iterace   1: x = 1.3572088083, změna = 1.4279e-01
Iterace   2: x = 1.3308609588, změna = 2.6348e-02
Iterace   3: x = 1.3258837742, změna = 4.9772e-03
Iterace   4: x = 1.3249393634, změna = 9.4441e-04
Iterace   5: x = 1.3247600113, změna = 1.7935e-04
Iterace   6: x = 1.3247259452, změna = 3.4066e-05
Iterace   7: x = 1.3247194745, změna = 6.4707e-06
Iterace   8: x = 1.3247182454, změna = 1.2291e-06
Iterace   9: x = 1.3247180120, změna = 2.3346e-07
Iterace  10: x = 1.3247179676, změna = 4.4345e-08
Iterace  11: x = 1.3247179592, změna = 8.4232e-09
Konvergoval v iteraci 11.
1.3247179592198772


In [ ]:
# ŘEŠENÍ
phi = lambda x: (x + 1)**(1/3)
koren = fixed_point_iteration(phi, 1.5, 1e-8, 200)
print(f'koren = {koren:.10f}')    # ≈ 1.3247179572
print(f'f(koren) = {koren**3 - koren - 1:.2e}')

### 1.6 Halleyho metoda

**Kdy:** máš i 2. derivaci. Konvergence kubická (rychlejší než Newton).  
**Cena:** musíš počítat f, f', f'' v každém kroku.

**Úkol:**  
Najdi kořen `f(x) = x³ - 2`.  
`f'(x) = 3x²`, `f''(x) = 6x`.  
Počáteční odhad `x0 = 1.5`, tolerance `1e-12`, max `30` iterací.  
Výsledek je ∛2.

In [53]:
# --- Halley ---
# Definuj f(x) = x³ - 2, f'(x) = 3x², f''(x) = 6x
# Zavolej halley(f, f1, f2, x0, tol, max_iter) s x0=1.5, tol=1e-12, max_iter=30



import math

def halley(f, f_prime, f_double, x0, tol, max_iter, verbose=False):
    """
    Hledání kořene funkce f(x)=0 pomocí Halleyho metody (využívá 1. a 2. derivaci).

    :param f: Funkce, jejíž kořen hledáme
    :param f_prime: První derivace funkce f
    :param f_double: Druhá derivace funkce f
    :param x0: Počáteční odhad kořene
    :param tol: Tolerance (přesnost)
    :param max_iter: Maximální počet iterací
    :param verbose: Vypisovat průběh iterací (default False)
    :return: Odhad kořene nebo None při selhání metody
    """
    if f is None or f_prime is None or f_double is None or x0 is None or tol is None or max_iter is None:
        print("Chyba: Vstupní hodnoty nesmí být None.")
        return None
    x = x0
    for i in range(max_iter):
        fx   = f(x)
        fpx  = f_prime(x)
        fdpx = f_double(x)
        denominator = 2 * (fpx ** 2) - fx * fdpx
        if abs(denominator) < 1e-12:
            print("Chyba: Nulový jmenovatel v Halleyho metodě.")
            return None
        x_new = x - (2 * fx * fpx) / denominator
        if math.isnan(x_new) or math.isinf(x_new):
            print("Chyba: Metoda divergovala (NaN/Inf).")
            return None

        if verbose:
            print(f"Iterace {i+1:3d}: x = {x_new:.10f}, f(x) = {f(x_new):.4e}, krok = {abs(x_new-x):.4e}")

        if abs(x_new - x) < tol or abs(f(x_new)) < tol:
            if verbose:
                print(f"Konvergoval v iteraci {i+1}.")
            return x_new
        x = x_new
    print("Chyba: Metoda neskonvergovala v daném počtu iterací.")
    return None



f =lambda x: x**3 -2 
fx1 = lambda x: 3*(x**2)
fx2 = lambda x: 6*x

x0 = 1.5
tol = 1e-12
iter = 30


y= 2
ocekavany_vysledek = y**(1/3)
print(ocekavany_vysledek)

vysledek = halley(f,fx1,fx2,x0,tol,iter)

if abs(vysledek - ocekavany_vysledek) < tol:
    print("Uspech")
else: 
    print("Neúspěch nebo špatná tolerance")





1.2599210498948732
Uspech


In [ ]:
# ŘEŠENÍ
f   = lambda x: x**3 - 2
f1  = lambda x: 3*x**2
f2  = lambda x: 6*x
koren = halley(f, f1, f2, 1.5, 1e-12, 30)
print(f'koren = {koren:.12f}')   # ≈ 1.259921049895
print(f'koren³ = {koren**3:.12f}')

### 1.7 Newton-Horner (kořen polynomu)

**Kdy:** hledáš kořen polynomu. Hornerovo schéma efektivně počítá P(x) a P'(x).  
**Koeficienty:** od nejnižší mocniny → `[a0, a1, a2, ...]` znamená `a0 + a1*x + a2*x² + ...`

**Úkol:**  
Polynom: `P(x) = -6 + 11x - 6x² + x³`  
Koeficienty (od nejnižší): `[-6, 11, -6, 1]`  
Kořeny tohoto polynomu jsou 1, 2, 3. Najdi ten kolem `x0 = 2.5`.  
Tolerance `1e-10`, max `100` iterací.

In [73]:
# --- Newton-Horner ---
# Polynom: P(x) = -6 + 11x - 6x² + x³  (kořeny: 1, 2, 3)
# Definuj koefs od nejnižší mocniny a najdi kořen kolem x0=2.5
# Zavolej newton_horner(coefs, x0, tol, max_iter), tol=1e-10, max_iter=100




import math

def newton_horner(coefs:list, x0, tol, max_iter, verbose=False):
    """
    Hledání kořene polynomu pomocí Newtonovy metody s Hornerovým schématem.

    :param coefs: Koeficienty polynomu od nejnižší po nejvyšší mocninu
    :param x0: Počáteční odhad kořene
    :param tol: Tolerance (přesnost)
    :param max_iter: Maximální počet iterací
    :param verbose: Vypisovat průběh iterací (default False)
    :return: Odhad kořene polynomu nebo None při selhání metody
    """
    if coefs is None or x0 is None or tol is None or max_iter is None:
        print("Chyba: Vstupní hodnoty nesmí být None.")
        return None
    if len(coefs) == 0:
        print("Chyba: Vstupní hodnoty nesmí být None.")
        return None
    if len(coefs) == 1:
        if abs(coefs[0]) < 1e-12:
            return x0
        else:
            print("Chyba: Konstantní polynom nemá kořen.")
            return None
    x = x0
    deg = len(coefs) - 1
    for iteration in range(max_iter):
        p_val = coefs[deg]
        p_der = 0.0
        for j in range(deg - 1, -1, -1):
            p_der = p_der * x + p_val
            p_val = p_val * x + coefs[j]
        if abs(p_der) < 1e-12:
            print("Chyba: Nulová derivace v metodě Newton-Horner.")
            return None
        x_new = x - p_val / p_der
        if math.isnan(x_new) or math.isinf(x_new):
            print("Chyba: Metoda divergovala (NaN/Inf).")
            return None

        if verbose:
            print(f"Iterace {iteration+1:3d}: x = {x_new:.10f}, P(x) = {p_val:.4e}, krok = {abs(x_new-x):.4e}")

        if abs(x_new - x) < tol or abs(p_val) < tol:
            if verbose:
                print(f"Konvergoval v iteraci {iteration+1}.")
            return x_new
        x = x_new
    print("Chyba: Metoda neskonvergovala v daném počtu iterací.")
    return None



import math


print('Importy OK')

f = lambda x: x**3 - 6*x**2 + 11*x - 6



print(f'{'x':>6}  {'f(x)':>12}  znaménko')
print('-' * 32)

prev_sign = None
for xi in [i * 0.5 for i in range(-4, 10)]:
    fx = f(xi)
    sign = '+' if fx > 0 else '-'
    zmena = '  ← ZMĚNA ZNAMÉNKA → kořen zde!' if (prev_sign and sign != prev_sign) else ''
    print(f'{xi:>6.1f}  {fx:>12.4f}  {sign}{zmena}')
    prev_sign = sign


coefs = [-6,11,-6,1]

x0 = 3.5

tol = 1e-10

iter = 100


vysledek = newton_horner(coefs, x0, tol, iter,verbose=True)

print(vysledek)











Importy OK
     x          f(x)  znaménko
--------------------------------
  -2.0      -60.0000  -
  -1.5      -39.3750  -
  -1.0      -24.0000  -
  -0.5      -13.1250  -
   0.0       -6.0000  -
   0.5       -1.8750  -
   1.0        0.0000  -
   1.5        0.3750  +  ← ZMĚNA ZNAMÉNKA → kořen zde!
   2.0        0.0000  -  ← ZMĚNA ZNAMÉNKA → kořen zde!
   2.5       -0.3750  -
   3.0        0.0000  -
   3.5        1.8750  +  ← ZMĚNA ZNAMÉNKA → kořen zde!
   4.0        6.0000  +
   4.5       13.1250  +
Iterace   1: x = 3.1739130435, P(x) = 1.8750e+00, krok = 3.2609e-01
Iterace   2: x = 3.0323071275, P(x) = 4.4382e-01, krok = 1.4161e-01
Iterace   3: x = 3.0014559538, P(x) = 6.7779e-02, krok = 3.0851e-02
Iterace   4: x = 3.0000031689, P(x) = 2.9183e-03, krok = 1.4528e-03
Iterace   5: x = 3.0000000000, P(x) = 6.3379e-06, krok = 3.1689e-06
Iterace   6: x = 3.0000000000, P(x) = 3.0128e-11, krok = 1.5064e-11
Konvergoval v iteraci 6.
3.0


In [ ]:
# ŘEŠENÍ
coefs = [-6, 11, -6, 1]   # P(x) = x³ - 6x² + 11x - 6
koren = newton_horner(coefs, 2.5, 1e-10, 100)
print(f'koren = {koren:.10f}')   # ≈ 3.0
# Zkus i x0=0.5 → najde kořen 1, x0=1.5 → najde kořen 2

---
## 2. Interpolace
### 2.1 Lineární interpolace po částech

**Kdy:** máš tabulku naměřených hodnot a chceš odhadnout hodnotu mezi body.  
**Princip:** na každém intervalu [x_i, x_{i+1}] je přímka.

**Úkol:**  
Teplota vzduchu v průběhu dne (každé 4 hodiny):

| Čas (h) | Teplota (°C) |
|---------|-------------|
| 0       | 8.0         |
| 4       | 6.5         |
| 8       | 11.0        |
| 12      | 18.5        |
| 16      | 20.0        |
| 20      | 15.0        |
| 24      | 10.0        |

Odhadni teplotu v čase `t = 10` hod, `t = 14` hod a `t = 22` hod.

In [ ]:
# --- Lineární interpolace ---
# Data jsou ze zadání výše — přepiš je
# Zavolej linear_interpolation(x_vals, y_vals, t) pro t = 10, 14, 22

def linear_interpolation(x_vals, y_vals, t):
    """
    Lineární interpolace po částech pro zadané body.

    :param x_vals: Seznam uzlů x (musí mít stejnou délku jako y_vals)
    :param y_vals: Seznam uzlů y (hodnot funkce v bodech x_vals)
    :param t: Bod nebo seznam bodů, ve kterých chceme interpolovat
    :return: Interpolovaná hodnota (nebo seznam hodnot) nebo None při chybě
    """
    if x_vals is None or y_vals is None or t is None:
        print("Chyba: Vstupní hodnoty nesmí být None.")
        return None
    n = len(x_vals)
    if n == 0 or n != len(y_vals):
        print("Chyba: Nesouhlasí délky vstupních seznamů.")
        return None
    # Pokud t není seznam, zabalíme ho do seznamu pro jednotné zpracování
    t_list = t if isinstance(t, list) else [t]
    x_min = x_vals[0]
    x_max = x_vals[-1]
    # Kontrola, zda všechny body t jsou v rozsahu [x_min, x_max]
    for val in t_list:
        if val < x_min or val > x_max:
            print("Chyba: Bod interpolace je mimo rozsah.")
            return None
    result = []
    for val in t_list:
        interpolated_val = None
        # Najdeme interval [x_i, x_{i+1}], v němž leží hodnota val
        for i in range(n - 1):
            if x_vals[i] <= val <= x_vals[i + 1]:
                if val == x_vals[i]:
                    interpolated_val = y_vals[i]
                elif val == x_vals[i + 1]:
                    interpolated_val = y_vals[i + 1]
                else:
                    interpolated_val = y_vals[i] + (y_vals[i+1] - y_vals[i]) * ((val - x_vals[i]) / (x_vals[i+1] - x_vals[i]))
                break
        if interpolated_val is None:
            # Pokud nastane (val == x_max), vezmeme poslední bod
            if val == x_max:
                interpolated_val = y_vals[-1]
            else:
                # Mimo rozsah (nemělo by nastat díky kontrole výše)
                print("Chyba: Bod interpolace je mimo rozsah.")
                return None
        result.append(interpolated_val)
    # Pokud vstup nebyl seznam, vracíme skalární hodnotu místo seznamu
    if not isinstance(t, list):
        return result[0] if result else None
    return result


hours = [0,4,8,12,16,20,24]
temperature = [8.0,6.5,11.0,18.5,20.0,15.0,10.0]



t10 = linear_interpolation(hours,temperature,10)
t14 = linear_interpolation(hours,temperature,14)
t22 = linear_interpolation(hours,temperature,22)

print(f't=10h → {t10} °C')
print(f't=14h → {t14} °C')
print(f't=22h → {t22} °C')





t=10h → 14.75 °C
t=14h → 19.25 °C
t=22h → 12.5 °C


In [ ]:
# ŘEŠENÍ
x_vals = [0, 4, 8, 12, 16, 20, 24]
y_vals = [8.0, 6.5, 11.0, 18.5, 20.0, 15.0, 10.0]
t10 = linear_interpolation(x_vals, y_vals, 10)
t14 = linear_interpolation(x_vals, y_vals, 14)
t22 = linear_interpolation(x_vals, y_vals, 22)
print(f't=10h → {t10:.3f} °C')    # ≈ 14.75
print(f't=14h → {t14:.3f} °C')    # ≈ 19.25
print(f't=22h → {t22:.3f} °C')    # ≈ 12.5

---
## 3. Soustavy lineárních rovnic
### 3.1 Gaussova eliminace

**Kdy:** přímá metoda, vždy najde řešení (pokud existuje). O(n³).  
**Pozor:** základní verze bez pivotace může selhat na nulový pivot.

**Úkol:**  
Vyřeš soustavu:
```
 2x +  y -  z =  8
-3x - y + 2z = -11
-2x + y + 2z = -3
```
Správné řešení: `x=2, y=3, z=-1`.

In [ ]:
# --- Gaussova eliminace ---
# Soustava:  2x +  y -  z =  8
#           -3x -  y + 2z = -11
#           -2x +  y + 2z = -3
# Definuj matici A a vektor b, zavolej gauss(A, b)

x = None
print(f'x = {x}')   # má být [2.0, 3.0, -1.0]

In [ ]:
# ŘEŠENÍ
A = [
    [ 2,  1, -1],
    [-3, -1,  2],
    [-2,  1,  2]
]
b = [8, -11, -3]
x = gauss(A, b)
print(f'x = {[round(v, 6) for v in x]}')

### 3.2 Gauss-Seidelova metoda

**Kdy:** iterativní metoda, vhodná pro velké řídké soustavy.  
**Podmínka konvergence:** diagonálně dominantní matice (|a_ii| > Σ|a_ij| pro j≠i).  
**Rozdíl od Jacobiho:** nové hodnoty x[i] se používají **ihned** v téže iteraci → rychlejší.

**Úkol:**  
Soustava (diagonálně dominantní):
```
10x +  y + 2z = 44
 x + 10y + 3z = 51
 2x +  3y + 10z = 61
```
Max `500` iterací, tolerance `1e-8`.  
Správné řešení: `x=1, y=2, z=4`.

In [ ]:
# --- Gauss-Seidel ---
# Soustava: 10x +  y + 2z = 44
#            x + 10y + 3z = 51
#           2x +  3y + 10z = 61
# Definuj A a b, zavolej gauss_seidel(A, b, max_iter, tol), max_iter=500, tol=1e-8

x = None
print(f'x = {x}')   # má být [1.0, 2.0, 4.0]

In [ ]:
# ŘEŠENÍ
A = [
    [10, 1, 2],
    [1, 10, 3],
    [2,  3, 10]
]
b = [44, 51, 61]
x = gauss_seidel(A, b, 500, 1e-8)
print(f'x = {[round(v, 6) for v in x]}')

### 3.3 Jacobiho metoda

**Kdy:** stejné jako Gauss-Seidel, ale nové hodnoty se **nepoužívají ihned** — aktualizuje se až po celé iteraci.  
**Výhoda:** snadno paralelizovatelná. **Nevýhoda:** pomalejší konvergence.

**Úkol:**  
Stejná soustava jako výše. Porovnej počet iterací s Gauss-Seidelem.  
Max `1000` iterací, tolerance `1e-8`.

In [ ]:
# --- Jacobi ---
# Stejná soustava jako Gauss-Seidel výše
# Zavolej jacobi(A, b, max_iter, tol), max_iter=1000, tol=1e-8

x = None
print(f'x = {x}')   # má být [1.0, 2.0, 4.0]

In [ ]:
# ŘEŠENÍ
A = [
    [10, 1, 2],
    [1, 10, 3],
    [2,  3, 10]
]
b = [44, 51, 61]
x = jacobi(A, b, 1000, 1e-8)
print(f'x = {[round(v, 6) for v in x]}')

---
## 4. Aproximace — metoda nejmenších čtverců (LSA)

**Kdy:** máš zašuměná naměřená data a chceš proložit polynomem stupně (n-1).  
**Princip:** minimalizuje součet čtverců odchylek Σ(y_i - P(x_i))².

**Úkol:**  
Naměřená data (průhyb nosníku v závislosti na zatížení):

| Zatížení (kN) | Průhyb (mm) |
|--------------|-------------|
| 1            | 2.1         |
| 2            | 4.0         |
| 3            | 6.2         |
| 4            | 7.9         |
| 5            | 10.1        |
| 6            | 12.0        |

Prolož **lineárním polynomem** (n=2, tj. `a0 + a1*x`).  
Pak zkus **kvadratický** (n=3). Porovnej koeficienty.

In [ ]:
# --- LSA ---
# Data jsou ze zadání výše — přepiš je
# Zavolej lsa(x_data, y_data, n) pro n=2 (lineární) a n=3 (kvadratický)

koefs_lin  = None
koefs_quad = None
print(f'Lineární:    {koefs_lin}')
print(f'Kvadratický: {koefs_quad}')

In [ ]:
# ŘEŠENÍ
x_data = [1, 2, 3, 4, 5, 6]
y_data = [2.1, 4.0, 6.2, 7.9, 10.1, 12.0]
koefs_lin  = lsa(x_data, y_data, 2)
koefs_quad = lsa(x_data, y_data, 3)
print(f'Lineární:    a0={koefs_lin[0]:.4f}, a1={koefs_lin[1]:.4f}')
x_pred = 7
y_lin  = koefs_lin[0] + koefs_lin[1]*x_pred
y_quad = koefs_quad[0] + koefs_quad[1]*x_pred + koefs_quad[2]*x_pred**2
print(f'Předpověď x=7: lineární={y_lin:.3f} mm, kvadratický={y_quad:.3f} mm')

---
## 6. Numerická integrace
### 6.1 Simpsonovo pravidlo

**Kdy:** přesná integrace hladkých funkcí. Na každých dvou intervalech fituje parabolu.  
**Podmínka:** `n` musí být **sudé** a ≥ 2.  
**Chyba:** O(h⁴) → mnohem přesnější než lichoběžník (O(h²)).

**Úkol A:**  
Spočítej `∫₀^π sin(x) dx`. Přesná hodnota = 2.  
Použij `n=10` dělení. Jaká je absolutní chyba?

**Úkol B:**  
Spočítej `∫₁^e (1/x) dx`. Přesná hodnota = 1 (protože ln(e)-ln(1)=1).  
Použij `n=100` dělení.

In [ ]:
# --- Simpson ---
# Úkol A: spočítej ∫₀^π sin(x) dx  (přesná hodnota = 2), n=10
# Úkol B: spočítej ∫₁^e (1/x) dx   (přesná hodnota = 1), n=100
# Zavolej simpson_rule(f, a, b, n)

I_A = None
I_B = None
print(f'A: I = {I_A}')
print(f'B: I = {I_B}')

In [ ]:
# ŘEŠENÍ
I_A = simpson_rule(math.sin, 0, math.pi, 10)
print(f'A: I = {I_A:.10f}, chyba = {abs(I_A - 2):.2e}')

I_B = simpson_rule(lambda x: 1/x, 1, math.e, 100)
print(f'B: I = {I_B:.10f}, chyba = {abs(I_B - 1):.2e}')

### 6.2 Rombergova metoda

**Kdy:** chceš vysokou přesnost bez ručního nastavování `n`.  
**Princip:** Richardson extrapolace na lichoběžníkové pravidlo — iterativně zdvojuje body a eliminuje chybu.  
**Výhoda:** automaticky konverguje na zadanou toleranci.

**Úkol:**  
Spočítej `∫₀^1 e^(-x²) dx` (Gaussův integrál — nemá analytické vyjádření).  
Referenční hodnota: `≈ 0.7468241328`.  
Tolerance `1e-10`, max `20` iterací.  

Pak zkus stejný integrál Simpsonem s `n=1000` — kdo je přesnější?

In [ ]:
# --- Romberg ---
# Spočítej ∫₀^1 e^(-x²) dx  (referenční hodnota: 0.7468241328124271)
# Zavolej romberg_quadrature(f, a, b, tol, max_iter), tol=1e-10, max_iter=20
# Pak stejný integrál Simpsonem s n=1000 — porovnej přesnost

I_romberg = None
I_simpson = None
print(f'Romberg: {I_romberg}')
print(f'Simpson: {I_simpson}')

In [ ]:
# ŘEŠENÍ
f = lambda x: math.exp(-x**2)
ref = 0.7468241328124271
I_romberg = romberg_quadrature(f, 0, 1, 1e-10, 20)
I_simpson = simpson_rule(f, 0, 1, 1000)
print(f'Romberg:  {I_romberg:.12f}, chyba = {abs(I_romberg - ref):.2e}')
print(f'Simpson:  {I_simpson:.12f}, chyba = {abs(I_simpson - ref):.2e}')

---
## 7. Diferenciální rovnice
### 7.1 Eulerova metoda

**Kdy:** nejjednodušší řešič ODE. Vhodný pro hrubý odhad nebo výukové účely.  
**Princip:** `y_{n+1} = y_n + h·f(x_n, y_n)`  
**Chyba:** O(h) — globální chyba roste lineárně s krokem h.

**Úkol:**  
ODE: `y' = y`, počáteční podmínka `y(0) = 1`.  
Analytické řešení: `y(x) = e^x`.  
Spočítej numericky na intervalu `[0, 1]` s krokem `h=0.1` (n=10 kroků).  
Porovnej `y(1)` numericky vs. `e¹ = 2.71828...`

In [ ]:
# --- Euler ---
# ODE: y' = y,  y(0) = 1  → analytické řešení y(x) = e^x
# Definuj f(x,y), x0, y0, h=0.1, n=10
# Zavolej euler_step(f, x0, y0, h, n)  → vrací seznam y[0]..y[n]

ys = None
print(f'Euler y(1) = {ys}')

In [ ]:
# ŘEŠENÍ
f  = lambda x, y: y
ys = euler_step(f, 0, 1, 0.1, 10)
print(f'Euler y(1)      = {ys[-1]:.8f}')   # ≈ 2.59374246
print(f'Přesná y(1)=e¹  = {math.e:.8f}')
print(f'Chyba           = {abs(ys[-1] - math.e):.4f}')   # ≈ 0.1245

### 7.2 Runge-Kutta 4. řádu (RK4)

**Kdy:** standardní metoda pro ODE. Zlatý standard přesnosti vs. výpočetní náklady.  
**Princip:** 4 evaluace f per krok, chyba O(h⁴) — mnohem přesnější než Euler.  
**Vrací:** seznam dvojic `[(x0, y0), (x1, y1), ...]`

**Úkol A:**  
Stejná ODE jako Euler: `y' = y`, `y(0) = 1`, h=0.1, n=10.  
Porovnej chybu RK4 vs. Euler pro `y(1)`.

**Úkol B:**  
ODE: `y' = -2xy`, `y(0) = 1`.  
Analytické řešení: `y(x) = e^(-x²)`.  
h=0.1, n=20. Porovnej `y(2)` numericky vs. přesně.

In [ ]:
# --- RK4 ---
# Úkol A: y' = y,  y(0) = 1,  h=0.1, n=10  → porovnej y(1) s e¹
# Úkol B: y' = -2xy,  y(0) = 1,  h=0.1, n=20  → porovnej y(2) s e^(-4)
# Zavolej runge_kutta_4(f, x0, y0, h, n)  → vrací [(x0,y0), (x1,y1), ...]

pts_A = None
pts_B = None
print(f'A: y(1) = {pts_A}')
print(f'B: y(2) = {pts_B}')

In [ ]:
# ŘEŠENÍ
pts_A = runge_kutta_4(lambda x, y: y, 0, 1, 0.1, 10)
y_rk4  = pts_A[-1][1]
print(f'A: RK4 y(1) = {y_rk4:.12f}')    # ≈ 2.718281828
print(f'A: Chyba RK4 = {abs(y_rk4 - math.e):.2e}')

pts_B = runge_kutta_4(lambda x, y: -2*x*y, 0, 1, 0.1, 20)
y_rk4_B  = pts_B[-1][1]
y_exac_B = math.exp(-4)
print(f'B: RK4 y(2)  = {y_rk4_B:.12f}')
print(f'B: Přesně    = {y_exac_B:.12f}')
print(f'B: Chyba RK4 = {abs(y_rk4_B - y_exac_B):.2e}')

---
## Shrnutí — Kdy co použít

| Problém | Metoda | Výhoda | Nevýhoda |
|---------|--------|--------|----------|
| Kořen f(x)=0, máš interval | Bisekce | Vždy konverguje | Pomalá (lineární) |
| Kořen f(x)=0, máš interval | Regula Falsi | Rychlejší než bisekce | Jeden konec fixní |
| Kořen f(x)=0, máš f' | Newton | Kvadratická konvergence | Nutná derivace, může divergovat |
| Kořen f(x)=0, bez f' | Steffensen | Kvadratická, bez derivace | Složitější implementace |
| Kořen f(x)=0, máš φ(x) | Fixed point | Jednoduchý | Konverguje jen pokud \|φ'\|<1 |
| Kořen f(x)=0, máš f,f',f'' | Halley | Kubická konvergence | Nutné 2 derivace |
| Kořen polynomu | Newton-Horner | Efektivní (Horner) | Jen pro polynomy |
| Hodnota mezi body | Lin. interpolace | Rychlá | Nespojitá derivace |
| Soustava Ax=b (přímá) | Gauss | Exaktní, vždy | O(n³) |
| Soustava Ax=b (iterační) | Gauss-Seidel | Rychlejší než Jacobi | Vyžaduje diag. dominanci |
| Soustava Ax=b (iterační) | Jacobi | Paralelizovatelný | Pomalejší než G-S |
| Proložení dat | LSA | Robustní na šum | Stupeň polynomu ad hoc |
| Integrál, hladká f | Simpson | Přesné, O(h⁴) | Sudý n |
| Integrál, vysoká přesnost | Romberg | Auto-adaptivní | Složitější |
| ODE, hrubý odhad | Euler | Nejjednodušší | Velká chyba O(h) |
| ODE, standard | RK4 | Přesné O(h⁴) | 4× víc evaluací f |